## What is Naive Bayes?

Naive Bayes is a **probabilistic classification algorithm** built directly on Bayes' Theorem. Given a set of features, I use it to compute the probability that a sample belongs to each possible class, then predict whichever class has the highest probability.

It's called "naive" because of a simplifying assumption I make: that all features are **conditionally independent** given the class. This assumption is almost never true in real data (features usually correlate with each other), but the algorithm works surprisingly well in practice anyway — especially for text classification, spam filtering, and other high-dimensional problems.

## Bayes' Theorem — The Foundation

$$P(y \mid X) = \frac{P(X \mid y) \cdot P(y)}{P(X)}$$

Breaking this down:
- $P(y \mid X)$ — **Posterior**: probability the sample belongs to class $y$, given the observed features $X$. This is what I actually want to compute.
- $P(X \mid y)$ — **Likelihood**: probability of seeing these particular feature values, given the sample belongs to class $y$.
- $P(y)$ — **Prior**: probability of class $y$ occurring at all, before looking at any features (just the overall class frequency in the data).
- $P(X)$ — **Evidence**: overall probability of seeing this feature combination, regardless of class. This acts as a normalizing constant.

## The "Naive" Independence Assumption

If $X$ consists of multiple features $x_1, x_2, ..., x_n$, computing the true joint likelihood $P(X \mid y)$ directly would require modeling how all the features interact with each other — computationally expensive and often infeasible with limited data.

Naive Bayes sidesteps this by assuming each feature is independent of the others, given the class:

$$P(X \mid y) = P(x_1, x_2, ..., x_n \mid y) \approx P(x_1 \mid y) \cdot P(x_2 \mid y) \cdot \, ... \, \cdot P(x_n \mid y)$$

This turns one hard joint-probability problem into several easy individual ones — each $P(x_i \mid y)$ can be estimated directly from frequency counts in the training data, exactly like the crosstabs I used in the tennis-playing example.

## Putting It Together — The Classification Rule

Since $P(X)$ is the same regardless of which class I'm evaluating, I don't actually need to compute it to compare classes — it cancels out. So the decision rule simplifies to:

$$\hat{y} = \arg\max_{y} \, P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

I compute this product for every possible class, then predict whichever class gives the highest score. This is exactly what happened in the tennis example — I multiplied the prior by each conditional probability for both "Yes" and "No," and picked whichever came out larger.

## Why Skip Normalization for Classification?

The raw products (like `0.0053` and `0.0206` from the tennis example) aren't true probabilities — they don't sum to 1, since I dropped the shared denominator $P(X)$. But for pure classification, that doesn't matter: I only care about *which* class scores higher, not the exact probability value. If I want an actual probability estimate (e.g., "there's an 80% chance this is spam"), I need to normalize by dividing each class's score by the sum of all classes' scores.

## Types of Naive Bayes (by feature type)

The core theorem stays the same, but how I estimate $P(x_i \mid y)$ changes depending on the feature type:

**Categorical Naive Bayes** — used in the tennis example. Features are discrete categories (Sunny/Rain/Overcast), and $P(x_i \mid y)$ is estimated directly as a frequency ratio from a crosstab, exactly as done manually.

**Gaussian Naive Bayes** — for continuous numerical features. Assumes each feature follows a normal distribution within each class, so $P(x_i \mid y)$ is computed using the Gaussian probability density function:

$$P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma_y^2}} \exp\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$$

where $\mu_y$ and $\sigma_y^2$ are the mean and variance of feature $x_i$ within class $y$, estimated from training data.

**Multinomial Naive Bayes** — commonly used for text classification (spam detection, document categorization), where features are word counts or frequencies. $P(x_i \mid y)$ is estimated based on how often a word appears in documents of a given class.

**Bernoulli Naive Bayes** — similar to Multinomial, but features are binary (word present/absent) rather than counts — useful when only presence matters, not frequency.

## The Zero-Frequency Problem — Laplace Smoothing

If a particular feature value never appears with a given class in the training data (e.g., no "Overcast" days had "No" for play in the tennis example — which is actually the case there), $P(x_i \mid y) = 0$. Since the whole prediction is a *product*, one zero wipes out the entire calculation regardless of how strong the other feature evidence is.

**Laplace Smoothing** fixes this by adding a small constant (usually 1) to every count before computing probabilities:

$$P(x_i \mid y) = \frac{\text{count}(x_i, y) + 1}{\text{count}(y) + k}$$

where $k$ is the number of possible values that feature can take. This ensures no probability is ever exactly zero, while barely affecting the estimate when there's already plenty of data.

## Advantages and Limitations

| Advantages | Limitations |
|---|---|
| Fast to train and predict — no iterative optimization needed | Independence assumption is rarely true in practice |
| Works well even with relatively small training data | Can perform poorly when features are strongly correlated |
| Handles high-dimensional data well (e.g., thousands of words in text classification) | Zero-frequency problem requires smoothing to handle properly |
| Naturally handles multi-class problems | Probability estimates themselves are often poorly calibrated, even when classification is correct |

## Practical Use — sklearn

```python
from sklearn.naive_bayes import GaussianNB, CategoricalNB, MultinomialNB

# For continuous features
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# For categorical features (like the tennis dataset)
cnb = CategoricalNB()
cnb.fit(X_train_encoded, y_train)  # categorical features need to be label-encoded first

y_pred = gnb.predict(X_test)
y_prob = gnb.predict_proba(X_test)  # normalized probabilities per class
```

For the tennis dataset specifically, `CategoricalNB` would replicate everything done manually in the notebook — but automatically, across all rows, with Laplace smoothing already built in by default.